# 13 · Clip polygons against shapely

**Question.** Does Fieldwork's `clip_polygons` widget intersect polygons with a boundary the
way GEOS does, and does it drop the ones that fall outside entirely?

Fieldwork intersects with **polyclip-ts**; the reference intersects with **GEOS**. Both work
on longitude and latitude directly, because that is what the widget declares — it does not
project first. The check compares what the widget claims, not what it might have claimed.

The fixture is the John Snow example's own chain: nearest-pump Voronoi cells clipped to the
reporting rectangle. The cells are computed in a finite envelope and extend past the
rectangle, so the clip is doing real work — an unclipped cell claims ground the report does
not cover.

Criteria fixed before running: the same polygons surviving, each within 1e-13 square degrees
of shapely's intersection by symmetric difference, and no output polygon extending beyond the
boundary. Tolerances are in square degrees because the operation is; at this latitude 1e-13
square degrees is under a square millimetre on the ground.

In [ ]:
# Only what this notebook imports. Requesting rasterio alongside pyproj kills this kernel
# outright; see 00-kernel-check for the measured map.
NEEDED = ["shapely", "pyproj", "numpy"]
try:
    import pyodide_js
    await pyodide_js.loadPackage(NEEDED)
    print("Requested from the Pyodide distribution:", ", ".join(NEEDED))
except ImportError:
    print("Not a Pyodide kernel; using the ambient environment.")
except Exception as error:
    print("loadPackage failed:", type(error).__name__, error)
    try:
        import piplite
        await piplite.install(NEEDED)
    except Exception as fallback:
        print("piplite also failed:", type(fallback).__name__, fallback)

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_13_clip_polygons import run

report = run(Path.cwd() / "fixtures" / "clip-polygons-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"polygons in            {report['polygonsIn']}")
print(f"survived, fieldwork    {report['polygonsOut']}")
print(f"survived, shapely      {report['referenceOut']}")
print(f"worst symmetric difference  {report['worstSymmetricDifferenceSquareDegrees']:.3e} square degrees")
print(f"polygons escaping the boundary  {report['escapedTheBoundary'] or 'none'}")
print()
for row in report["perPolygon"]:
    print(f"  {row['id']:<12} {row['symmetricDifferenceSquareDegrees']:.3e}")

## Reading the result

`agrees` means polyclip-ts and GEOS intersected the same polygons to the same ground. It
does not mean clipping to a reporting rectangle is the right thing to do: a clipped catchment
is the part of that catchment the report covers, which is a different object from the
catchment, and both implementations produce it identically.